## Fine-tuning MiniLM

Full fine-tune of `all-MiniLM-L6-v2` (every parameter gets gradients, not just a
classification head) on the same raw-text, same-split data as
`SMS_Spam_Classifier_FrozenEmbeddings.ipynb`. This notebook is written to run on a GPU
(Google Colab's free tier is enough) - fine-tuning 5 seeds x 6 epochs takes well under a
minute there, versus an estimated 10-30 minutes per run on this machine's CPU. It isn't
executed in this repo for that reason; see the project README for the actual results from
running it on Colab.

To run it: open in Colab, upload or `git clone` to get `Dataset_5971.csv`, `!pip install
transformers`, run top to bottom.

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import copy
import shutil
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [ ]:
DATASET_FILE = "Dataset_5971.csv"

df = pd.read_csv(DATASET_FILE)
df["TEXT"] = df["TEXT"].str.lstrip("\t")
df["CLEAN"] = df["TEXT"].str.lower().str.split().str.join(" ")
df["LABEL"] = df["LABEL"].str.strip().str.lower()
df = df.drop_duplicates(subset="CLEAN").drop(columns="CLEAN")

X = df["TEXT"]
y = df["LABEL"]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)

encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_val_encoded = encoder.transform(y_val)
y_test_encoded = encoder.transform(y_test)

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_token_lengths = X_train.apply(lambda t: len(tokenizer.encode(t, add_special_tokens=True)))
max_length = int(np.percentile(train_token_lengths, 95))
max_length

In [ ]:
def tokenize(texts):
    return tokenizer(texts.tolist(), truncation=True, max_length=max_length)

train_encodings = tokenize(X_train)
val_encodings = tokenize(X_val)
test_encodings = tokenize(X_test)

class SMSDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = SMSDataset(train_encodings, y_train_encoded)
val_dataset = SMSDataset(val_encodings, y_val_encoded)
test_dataset = SMSDataset(test_encodings, y_test_encoded)

collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collator)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

counts = torch.bincount(torch.tensor(y_train_encoded), minlength=3).float()
class_weights = (counts.sum() / (3 * counts)).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

epochs = 6
seeds = [0, 1, 2, 3, 4]
results = []

for seed in seeds:
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    best_f1, best_state = 0, None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            outputs = model(**batch)
            loss = loss_fn(outputs.logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        model.eval()
        preds, trues = [], []
        with torch.inference_mode():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("labels")
                outputs = model(**batch)
                preds.extend(outputs.logits.argmax(dim=1).tolist())
                trues.extend(labels.tolist())
        val_f1 = f1_score(trues, preds, average="macro")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

        print(f"seed {seed} epoch {epoch+1}/{epochs} loss {total_loss/len(train_loader):.4f} val macro-F1 {val_f1:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    preds, trues = [], []
    with torch.inference_mode():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            outputs = model(**batch)
            preds.extend(outputs.logits.argmax(dim=1).tolist())
            trues.extend(labels.tolist())

    test_f1 = f1_score(trues, preds, average="macro")
    results.append({"seed": seed, "test_macro_f1": test_f1, "state_dict": best_state, "preds": preds, "trues": trues})
    print(f"seed {seed} test macro-F1 {test_f1:.4f}")

In [ ]:
f1_scores = [r["test_macro_f1"] for r in results]
print(np.mean(f1_scores), np.std(f1_scores))

best_result = max(results, key=lambda r: r["test_macro_f1"])
print(classification_report(best_result["trues"], best_result["preds"], target_names=encoder.classes_, digits=3))
print(confusion_matrix(best_result["trues"], best_result["preds"]))

## Saving a checkpoint (optional)

Not committed to this repo - the trained weights are ~90MB and regenerating them is a
one-minute Colab run, not something worth versioning. Uncomment to save/download the best
seed's checkpoint if you want to use it outside this notebook.

In [ ]:
# model.load_state_dict(best_result["state_dict"])
# model.save_pretrained("minilm-finetuned")
# tokenizer.save_pretrained("minilm-finetuned")
# shutil.make_archive("minilm-finetuned", "zip", "minilm-finetuned")
# from google.colab import files
# files.download("minilm-finetuned.zip")